In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import sys
import time
from pathlib import Path
from typing import Dict, List, Any, Optional
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, classification_report, confusion_matrix
)
import mlflow

# Plotting configuration
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Add src to path for imports
sys.path.append(str(Path.cwd().parent))

# Import custom modules
from src.utils.config import load_config, setup_logging
from src.data.make_dataset import load_raw_data
from src.evaluation.cross_validation import create_stratified_kfold_splits
from src.models.baseline_model import BaselineModel
from src.models.gradient_boosting.xgboost_model import XGBoostModel
from src.models.gradient_boosting.lightgbm_model import LightGBMModel
from src.models.gradient_boosting.catboost_model import CatBoostModel
from src.models.neural_network_model import NeuralNetworkModel
from src.models.ensemble_model import EnsembleModel
from src.evaluation.metrics import calculate_comprehensive_metrics
from src.visualization.model_plots import plot_confusion_matrix, plot_roc_curves

print("✅ Libraries imported successfully!")

# Set random seeds for reproducibility
np.random.seed(42)

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)


In [ ]:
# Load configuration
try:
    config = load_config('config/config.yaml')
    print("✅ Configuration loaded successfully!")
except FileNotFoundError:
    # Create a comprehensive config for demonstration
    config = {
        'data': {
            'train_file': 'train.csv',
            'test_file': 'test.csv'
        },
        'target_column': 'Stay',
        'models': {
            'baseline': {'strategy': 'most_frequent'},
            'xgboost': {
                'n_estimators': 100,
                'max_depth': 6,
                'learning_rate': 0.1,
                'random_state': 42
            },
            'lightgbm': {
                'n_estimators': 100,
                'max_depth': 6,
                'learning_rate': 0.1,
                'random_state': 42
            },
            'catboost': {
                'iterations': 100,
                'depth': 6,
                'learning_rate': 0.1,
                'random_state': 42,
                'verbose': False
            }
        },
        'cross_validation': {
            'n_splits': 5,
            'random_state': 42
        },
        'mlflow': {
            'experiment_name': 'Healthcare_LOS_Prediction'
        }
    }
    print("⚠️  Using default configuration")

# Setup MLflow experiment
experiment_name = config.get('mlflow', {}).get('experiment_name', 'Healthcare_LOS_Prediction')
mlflow.set_experiment(experiment_name)
print(f"🧪 MLflow experiment: {experiment_name}")

# Load data
print("\n📂 Loading training data...")
try:
    train_df, _ = load_raw_data(config)
    print(f"✅ Training data loaded: {train_df.shape}")
    
    # Prepare features and target
    target_col = config.get('target_column', 'Stay')
    if target_col in train_df.columns:
        X = train_df.drop(columns=[target_col, 'case_id'], errors='ignore')
        y = train_df[target_col]
    else:
        # If target column not found, create sample data
        raise KeyError(f"Target column '{target_col}' not found")
        
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("📝 Creating sample data for demonstration...")
    
    # Create sample healthcare data
    np.random.seed(42)
    n_samples = 1000
    train_df = pd.DataFrame({
        'age': np.random.normal(65, 15, n_samples),
        'gender': np.random.choice([0, 1], n_samples),
        'admission_type': np.random.choice([0, 1, 2], n_samples),
        'diagnosis_code': np.random.choice([0, 1, 2, 3], n_samples),
        'num_procedures': np.random.poisson(2, n_samples),
        'Stay': np.random.choice([0, 1, 2, 3, 4], n_samples, p=[0.3, 0.25, 0.2, 0.15, 0.1])
    })
    
    X = train_df.drop(columns=['Stay'])
    y = train_df['Stay']

print(f"\n📊 Data prepared:")
print(f"   • Features shape: {X.shape}")
print(f"   • Target shape: {y.shape}")
print(f"   • Target distribution: {y.value_counts().to_dict()}")
print(f"   • Feature columns: {list(X.columns)}")
